# DistilGPT2: Alpaca fine-tuning + MLflow serve

Hive: `pysparktest.alpaca`. Сначала выполните DAG `alpaca_postgres_to_hive`.

Notebook обучает `distilgpt2` на Alpaca-формате, регистрирует модель как `distilgpt-causal-lm` в MLflow Production и проверяет serve на порту `5002`.

После открытия: **Kernel → Restart** и **Run All**.

In [ ]:
# Если образ Jupyter старый, раскомментируйте и перезапустите kernel:
# %pip install -q "transformers>=4.36,<5" "accelerate>=1.1.0" torch

In [ ]:
%matplotlib inline

import importlib
import importlib.util
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from pyspark.sql import functions as F

_utils = Path("/home/jovyan/work/alpaca_llm_utils.py")
if _utils.is_file():
    _spec = importlib.util.spec_from_file_location("alpaca_llm_utils", _utils)
    alu = importlib.util.module_from_spec(_spec)
    sys.modules["alpaca_llm_utils"] = alu
    _spec.loader.exec_module(alu)
else:
    import alpaca_llm_utils as alu
    alu = importlib.reload(alu)

from hive_ddl_utils import create_hive_spark_session, read_alpaca_hive, wait_hive_ports

In [ ]:
wait_hive_ports()
spark = create_hive_spark_session(app_name="distilgpt-alpaca-eda")
sdf = read_alpaca_hive(spark)
stats_df = alu.with_text_lengths(sdf).cache()
print(f"Строк: {stats_df.count():,}")

In [ ]:
print("=== EDA summary ===")
for k, v in alu.eda_summary_spark(stats_df):
    print(f"  {k}: {v}")

stats_df.select(
    "instruction", "input", "output", "instruction_len", "output_len", "has_input"
).show(3, truncate=70)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), ["instruction_len", "input_len", "output_len", "text_len"]):
    bins, cnt = alu.length_histogram(stats_df, col)
    ax.bar(bins, cnt, width=45, align="edge", color="steelblue")
    ax.set_title(col)
plt.tight_layout()
plt.show()

## Fine-tuning DistilGPT2

База: `distilgpt2`. Это небольшая causal LM, поэтому notebook обучает full model без LoRA и сохраняет веса в `models/alpaca-distilgpt2`, которые затем монтируются в контейнер serve на порт `5002`.

In [ ]:
alu = importlib.reload(alu)

MAX_TRAIN_SAMPLES = 500
MAX_LENGTH = 256
MAX_STEPS = 50
BATCH_SIZE = 2
CPU_THREADS = 6
MODEL_NAME = alu.DEFAULT_DISTILGPT2_MODEL
OUTPUT_DIR = alu.DEFAULT_DISTILGPT2_OUTPUT_DIR
EXPERIMENT_NAME = "distilgpt_alpaca_finetune"
REGISTERED_MODEL_NAME = "distilgpt-causal-lm"

train_sdf, eval_sdf = alu.build_train_eval_spark(
    stats_df,
    max_samples=MAX_TRAIN_SAMPLES,
    test_frac=0.1,
    seed=42,
)

prompt_sample_row = stats_df.select(
    "text", "instruction", "input", "output"
).limit(1).collect()[0]

try:
    stats_df.unpersist()
except Exception:
    pass

print(f"train: {train_sdf.count():,}, eval: {eval_sdf.count():,}")
print(f"model: {MODEL_NAME}")
print(f"output_dir: {OUTPUT_DIR}")
print(f"max_steps: {MAX_STEPS}")
print(f"cpu_threads: {CPU_THREADS}")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_tok, eval_tok = alu.spark_texts_to_tokenized(
    train_sdf,
    eval_sdf,
    tokenizer,
    max_length=MAX_LENGTH,
    model_name=MODEL_NAME,
)

# Пары для метрик serve собираем до spark.stop().
eval_prompt_pairs = alu.collect_eval_prompt_pairs(eval_sdf, max_samples=20)
print(f"tokenized: train={len(train_tok)}, eval={len(eval_tok)}")
print(f"eval pairs for serve metrics: {len(eval_prompt_pairs)}")

try:
    stats_df.unpersist()
except Exception:
    pass
spark.stop()

In [ ]:
trainer, run_id = alu.train_causal_lm(
    train_tok,
    eval_tok,
    model_name=MODEL_NAME,
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    experiment_name=EXPERIMENT_NAME,
    register_model_name=REGISTERED_MODEL_NAME,
    register_stage="Production",
    use_lora=False,
    gradient_checkpointing=False,
    trust_remote_code=False,
    cpu_threads=CPU_THREADS,
)

## MLflow Registry и serve

- UI: http://localhost:5000 → Models → `distilgpt-causal-lm` (Production)
- Serve: http://localhost:5002 через compose service `mlflow-serve-alpaca`
- После обучения/перерегистрации: `docker compose up -d mlflow-serve-alpaca --force-recreate`

In [ ]:
import importlib
import mlflow_utils as mu

mu = importlib.reload(mu)

print(f"MLflow UI: {mu.mlflow_ui_url()} → Models → {mu.DISTILGPT_REGISTERED_MODEL_NAME}")
print(f"Serve:     {mu.distilgpt_serve_ui_url()}/invocations")
if "run_id" in globals():
    print(f"run_id:    {run_id}")
print("Restart serve after training:")
print("docker compose up -d mlflow-serve-alpaca --force-recreate")

## Проверка генерации через serve `:5002`

In [ ]:
if "prompt_sample_row" not in globals():
    raise RuntimeError("Сначала выполните ячейки подготовки данных и токенизации.")

row = prompt_sample_row
prompt = alu.prompt_for_generation_from_row(row)

print("=== Instruction ===")
print(row.instruction[:300])
if row.input:
    print("=== Input ===")
    print(row.input[:200])
print("\n=== Эталон ===")
print(row.output[:400])

In [ ]:
# Запустите в терминале после обучения, если serve еще не пересоздан:
# docker compose up -d mlflow-serve-alpaca --force-recreate

uri = mu.wait_for_distilgpt_serve(max_wait_sec=600)
print(uri)
print(mu.predict_distilgpt_via_serve(prompt, max_new_tokens=120, serve_uri=uri, timeout=600))

## Метрики генерации serve

ROUGE-L, chrF, `refusal_rate`, средняя длина ответа — сравнение с эталоном `output` на eval-подвыборке.

In [ ]:
# %pip install -q rouge-score sacrebleu  # если в образе Jupyter этих пакетов нет

if "eval_prompt_pairs" not in globals() or not eval_prompt_pairs:
    raise RuntimeError("Нет eval_prompt_pairs: выполните ячейку tokenization до spark.stop().")

uri = mu.wait_for_distilgpt_serve(max_wait_sec=600)
serve_metrics = mu.evaluate_deployed_distilgpt(
    eval_prompt_pairs,
    serve_uri=uri,
    max_samples=20,
    max_new_tokens=128,
    timeout=600,
    show_examples=3,
    mlflow_log=True,
    experiment_name=EXPERIMENT_NAME,
    run_name="distilgpt_serve_generation_eval",
)
serve_metrics